# Reproducing "Asset Pricing in Pre-trained Transformers" (arXiv:2505.01575)

This notebook walks through building, training (on synthetic debug data by default), evaluating, and backtesting the paper's `P_Trans_H3` model (pre-trained Transformer, 3 heads) — the paper's best-performing model across periods (Table 3).

See `README.md` for real-data setup instructions (`data/README_data.md`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
import pandas as pd
import torch

from sert_asset_pricing.utils.config import ConfigLoader, set_seed, get_device
from sert_asset_pricing.models.transformer_variants import build_model
from sert_asset_pricing.data.dataset import RollingFactorDataset, RollingWindowSpec
from sert_asset_pricing.data.transforms import FactorPreprocessor
from sert_asset_pricing.training.trainer import RollingWindowTrainer
from sert_asset_pricing.evaluation.metrics import OOSMetrics
from sert_asset_pricing.evaluation.backtest import Backtester

## 1. Load config

In [ ]:
cfg = ConfigLoader('../configs/config.yaml').load()
set_seed(cfg['training']['seed'])
device = get_device(cfg['hardware']['device'])
print('family:', cfg['model']['family'], '| device:', device)

## 2. Synthetic data (swap for real CSVs per data/README_data.md)

In [ ]:
n_dates = 200
dates = pd.date_range('2000-01-01', periods=n_dates, freq='MS')
factors = pd.DataFrame(torch.randn(n_dates, 20).numpy(), index=dates)
returns = pd.DataFrame(torch.randn(n_dates, 10).numpy() * 0.05, index=dates)

pre = FactorPreprocessor(missing_value_threshold=cfg['data']['missing_value_threshold'])
factors_clean = pre.fit_transform(factors)
factors_clean.shape

## 3. Build rolling-window datasets (small windows for notebook speed)

In [ ]:
cfg['model']['input_factor_dim'] = factors_clean.shape[1]
cfg['model']['num_heads'] = 3
cfg['model']['d_model'] = 30  # multiple of num_heads, kept small for notebook speed
cfg['training']['train_window'] = 40
cfg['training']['val_window'] = 10
cfg['training']['max_epochs'] = 5

spec = RollingWindowSpec(cfg['training']['train_window'], cfg['training']['val_window'], cfg['training']['step_size'])
train_ds = RollingFactorDataset(factors_clean, returns, spec=spec, split='train')
val_ds = RollingFactorDataset(factors_clean, returns, spec=spec, split='val')
len(train_ds), len(val_ds)

## 4. Build model and train

In [ ]:
model = build_model(cfg).to(device)
print(model)
trainer = RollingWindowTrainer(model, train_ds, val_ds, cfg, checkpoint_dir='../checkpoints', device=device)
history = trainer.fit()
history['train_loss'][-1], history['val_loss'][-1]

## 5. OOS R2 (Appendix E corrected methodology)

In [ ]:
import numpy as np
out = trainer.evaluate_oos('val')
pred = out['predictions'][0].numpy().squeeze()
target = out['targets'][0].numpy().squeeze()
hist_mean = returns.mean(axis=0).to_numpy()[0]
metrics = OOSMetrics()
r2 = metrics.oos_r2(target, pred, hist_mean)
print('OOS R2 (single stock, synthetic data):', r2)

## 6. Backtest (sign-signal, equal-weighted, static transaction cost)

In [ ]:
rng = np.random.default_rng(42)
preds = rng.normal(0, 0.03, size=(60, 20))
actuals = rng.normal(0, 0.05, size=(60, 20))
bt = Backtester(cfg['evaluation']['transaction_cost_static_bps'], cfg['evaluation']['transaction_cost_dynamic_bps'], cfg['evaluation']['softmax_filter_pct'])
result = bt.run(preds, actuals, weighting='equal', tc_mode='static')
{k: v for k, v in result.items() if k not in ('portfolio_returns', 'cumulative_returns')}

## Next steps
- Swap synthetic data for real factor/return CSVs (see `data/README_data.md`).
- Run `python train.py --config configs/config.yaml` (full paper-scale training).
- Sweep `model.family` and `model.num_heads` per Table 2 to reproduce the full model grid.